# CNN + LSTM Video Action Recognition in PyTorch
End-to-end video action recognition pipeline using:
- ResNet18 for frame feature extraction
- LSTM for temporal modeling
- PyTorch custom Dataset and DataLoader
- verify end to end flow by overfitting on a tiny dataset

##Imports




In [1]:
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

## Sample Videos
Three small sample videos are downloaded and used as a toy dataset for experimentation and building the training pipeline.

In [2]:
!mkdir data

# Video 1
!wget -P data https://samplelib.com/lib/preview/mp4/sample-5s.mp4

# Video 2
!wget -P data https://samplelib.com/lib/preview/mp4/sample-10s.mp4

# Video 3
!wget -P data https://samplelib.com/lib/preview/mp4/sample-15s.mp4

--2026-05-21 05:32:31--  https://samplelib.com/lib/preview/mp4/sample-5s.mp4
Resolving samplelib.com (samplelib.com)... 188.227.59.182
Connecting to samplelib.com (samplelib.com)|188.227.59.182|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://samplelib.com/preview/mp4/sample-5s.mp4 [following]
--2026-05-21 05:32:32--  https://samplelib.com/preview/mp4/sample-5s.mp4
Reusing existing connection to samplelib.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 2848208 (2.7M) [video/mp4]
Saving to: ‘data/sample-5s.mp4’

sample-5s.mp4       100%[===================>]   2.72M  4.16MB/s    in 0.7s    

2026-05-21 05:32:32 (4.16 MB/s) - ‘data/sample-5s.mp4’ saved [2848208/2848208]

--2026-05-21 05:32:32--  https://samplelib.com/lib/preview/mp4/sample-10s.mp4
Resolving samplelib.com (samplelib.com)... 188.227.59.182
Connecting to samplelib.com (samplelib.com)|188.227.59.182|:443... connected.
HTTP request sent, awaiting response... 30

## Custom Video Dataset


In [3]:
transform = transforms.Compose([transforms.ToTensor()])

class VideoDataset(Dataset):
  def __init__(self, samples):
    self.samples = samples

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    video_path, label = self.samples[index]
    frames = []
    cap = cv2.VideoCapture(video_path)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    frame_indices = np.linspace(0, total_frames-1, 3, dtype = int)
    for f in frame_indices:
      cap.set(cv2.CAP_PROP_POS_FRAMES, f-1)
      success, frame = cap.read()

      if not success:
        break

      else:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame= cv2.resize(frame, (224,224))
        frame = transform(frame)
        frames.append(frame)

    cap.release()
    clip = torch.stack(frames)

    return clip, label


## CNN + LSTM Video Model

- Pretrained ResNet18 for frame-level feature extraction
- LSTM for temporal sequence modeling
- Linear classifier for final prediction

In [4]:
class VideoModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.resnet = models.resnet18(pretrained = True)
    self.resnet.fc = nn.Identity()
    self.lstm = torch.nn.LSTM(input_size=512, hidden_size=256, batch_first=True)
    self.classifier = torch.nn.Linear(256, 3)

  def forward(self, input_clips):
    batch, frames, channel, H, W = input_clips.shape
    reshaped = input_clips.view(batch*frames, channel, H, W)

    features = self.resnet(reshaped)
    features = features.view(batch, frames, 512 )

    lstm_out, (hidden, cell) = self.lstm(features)

    final_hidden = hidden[-1]
    logits = self.classifier(final_hidden)
    return logits

## Dataset Preparation

In [5]:
samples = [
    ("data/sample-5s.mp4", 0),
    ("data/sample-10s.mp4", 1),
    ("data/sample-15s.mp4", 2)
]

dataset = VideoDataset(samples)
loader = DataLoader(dataset, batch_size =2, shuffle = True)

## Model Initialization

In [ ]:
model = VideoModel()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

##Training

In [7]:
model.train()
for epoch in range(7):
  epoch_loss = 0
  for clips, labels in loader:
    # print(labels)
    optimizer.zero_grad()

    logits = model(clips)
    loss =loss_fn(logits, labels)
    loss.backward()
    optimizer.step()
    # print(loss.item())

    epoch_loss += loss.item()
  print("epoch: ", epoch, "Loss: ", epoch_loss/len(loader))

epoch:  0 Loss:  1.674359679222107
epoch:  1 Loss:  0.7417791336774826
epoch:  2 Loss:  0.7120633125305176
epoch:  3 Loss:  0.3491622656583786
epoch:  4 Loss:  0.7036464810371399
epoch:  5 Loss:  0.20967985689640045
epoch:  6 Loss:  0.1312398798763752


## Inference

In [8]:
clips, labels = next(iter(loader))
with torch.no_grad():
  logits = model(clips)
  predictions = torch.argmax(logits, dim = 1)

  print("predictions= ", predictions)
  print("labels= ", labels)

predictions=  tensor([0, 1])
labels=  tensor([0, 1])


## Results
The model successfully overfit the tiny dataset